# 08 - Modelling Preparation

### Setup

In [1]:
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton unsloth_zoo
!pip install --no-deps cut_cross_entropy unsloth
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.9.2 requires structlog>=24.1.0, which is not installed.
unsloth 2026.9.2 requires tyro, which is not installed.
unsloth-zoo 2026.9.1 requires msgspec, which is not installed.
unsloth-zoo 2026.9.1 requires tyro; sys_platform != "darwin" or platform_machine != "ar

In [2]:
import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

## 1. Konfigurasi

In [3]:
# Ambil HF token dari Kaggle Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_Faiss")
 
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = "/kaggle/working"
 
MODEL_CONFIGS = {
    "llama3.2": {
        "hf_model": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "system_role": True,
    },
    "gemma2": {
        "hf_model": "unsloth/gemma-2-2b-it-bnb-4bit",
        "system_role": False,
    },
}
 
SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum menjawab."
)

## 2. Load dataset

In [4]:
# Load satu split dataset (train/validation) dari file CSV di repo HF Hub
def load_split(split_name):
    filename = "train.csv" if split_name == "train" else "val.csv"
    ds = load_dataset("csv", data_files=f"hf://datasets/{HF_REPO}/{filename}", split="train", token=HF_TOKEN)
    return ds

In [5]:
train_ds = load_split("train")
val_ds = load_split("validation")
print(f"train: {len(train_ds)} baris | validation: {len(val_ds)} baris")

train.csv:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

val.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

train: 6516 baris | validation: 1043 baris


In [6]:
df_train = train_ds.to_pandas()
df_train

,record_id,task_type,subtask_type,split,instruction,cot,response,context_joined,n_context_spans,document_title,document_year,section_id,section_title,context_chunk_ids,source_pages,bps_url,track_a_score,numeric_score
0,1035f635fb8e9f52,qa,factual_retrieval,train,Berapa persentase pelaku usaha yang pernah men...,"Teks menyatakan pada tahun 2020 tercatat 19,97...","Pada tahun 2020, persentase pelaku usaha yang ...","Pada periode tahun 2020, tercatat bahwa 19,97 ...",1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_101,5.3 Membayar Melebihi Ketentuan,analisis-petty-corruption-hasil-spak-2020---20...,143 ||| 144 ||| 145 ||| 146,https://www.bps.go.id/id/publication/2025/12/3...,1.00,1.0
1,0fe915c3f72d64c3,qa,source_navigation,train,Gambar nomor berapa yang menyajikan persentase...,Teks menyebut judul gambar yang memuat persent...,Persentase masyarakat yang pernah berurusan de...,Gambar 5.2 Persentase Masyarakat yang Pernah B...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_101,5.3 Membayar Melebihi Ketentuan,analisis-petty-corruption-hasil-spak-2020---20...,143 ||| 144 ||| 145 ||| 146,https://www.bps.go.id/id/publication/2025/12/3...,1.00,1.0
2,841b08bae21a32b7,qa,source_navigation,train,Pada gambar nomor berapa data mengenai persent...,Teks menyebut judul gambar yang memuat persent...,Data persentase pelaku usaha yang pernah berur...,Gambar 5.3 Persentase Pelaku Usaha yang Pernah...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_101,5.3 Membayar Melebihi Ketentuan,analisis-petty-corruption-hasil-spak-2020---20...,143 ||| 144 ||| 145 ||| 146,https://www.bps.go.id/id/publication/2025/12/3...,1.00,1.0
3,c649281e7ee666fb,qa,methodology_explanation,train,Bagaimana cara Indikator SDGs 16.5.1 mengukur ...,Teks menyatakan bahwa Indikator 16.5.1 menguku...,Indikator SDGs 16.5.1 diukur berdasarkan seber...,Indikator 16.5.1 berfokus pada pengalaman masy...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_101,5.3 Membayar Melebihi Ketentuan,analisis-petty-corruption-hasil-spak-2020---20...,143 ||| 144 ||| 145 ||| 146,https://www.bps.go.id/id/publication/2025/12/3...,1.00,1.0
4,f1471ab80a777315,qa,methodology_explanation,train,Bagaimana cara mengukur penyuapan pada lingkun...,Teks menyatakan bahwa indikator ini menghitung...,Pengukuran pada Indikator SDGs 16.5.2 dilakuka...,Indikator ini menghitung seberapa sering perus...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_101,5.3 Membayar Melebihi Ketentuan,analisis-petty-corruption-hasil-spak-2020---20...,143 ||| 144 ||| 145 ||| 146,https://www.bps.go.id/id/publication/2025/12/3...,0.55,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6511,d5f6975b4d8250d8,qa,definition_clarification,train,Apa yang dimaksud dengan Susenas Maret dan apa...,"Teks menyatakan sumber data adalah ""Survei Sos...",Susenas Maret adalah Survei Sosial Ekonomi Nas...,"Sumber / Source : Badan Pusat Statistik, Surve...",1,Statistik Kesejahteraan Rakyat 2024,2024,sec_132,Tabel Persentase Rumah Tangga di Daerah Perdes...,statistik-kesejahteraan-rakyat-2024__sec_132__...,236 ||| 237,https://www.bps.go.id/id/publication/2024/10/2...,1.00,1.0
6512,bc83011f1ed21bfe,qa,methodology_explanation,train,Bagaimana pengklasifikasian wilayah desa dan k...,"Teks menyatakan pembagian wilayah menjadi ""Per...",Data disajikan menurut klasifikasi wilayah Per...,Perkotaan/ Urban Perdesaan/ Rural Perkotaan+Pe...,1,Statistik Kesejahteraan Rakyat 2024,2024,sec_132,Tabel Persentase Rumah Tangga di Daerah Perdes...,statistik-kesejahteraan-rakyat-2024__sec_132__...,236 ||| 237,https://www.bps.go.id/id/publication/2024/10/2...,1.00,1.0
6513,2e87bc37629ecd69,qa,methodology_explanation,train,Bagaimana metode pengumpulan data yang digunak...,"Teks menyatakan sumber data berasal dari ""Surv...",Data dikumpulkan melalui Survei Sosial Ekonomi...,"Sumber / Source : Badan Pusat Statistik, Surve...",1,Statistik Kesejahteraan Rakyat 2024,2024,sec_132,Tabel Persentase Rumah Tangga 

## 3. Format Chat

In [7]:
# Gabungkan context_joined + instruction jadi satu blok teks user turn
def build_user_content(example):
    context_block = example["context_joined"]
    return f"Konteks:\n{context_block}\n\nPertanyaan: {example['instruction']}"
 
 
# Gabungkan cot + response jadi satu blok teks assistant turn (target loss)
def build_assistant_content(example):
    return f"{example['cot']}\n\n{example['response']}"
 
 
# Susun list messages format chat, menyesuaikan apakah arsitektur punya native system role
def to_chat_messages(example, system_role):
    user_content = build_user_content(example)
    assistant_content = build_assistant_content(example)
    if system_role:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    else:
        messages = [
            {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_content}"},
            {"role": "assistant", "content": assistant_content},
        ]
    return messages

In [8]:
# Terapkan chat template tokenizer ke seluruh dataset, hasilkan kolom "text"
def format_dataset(dataset, tokenizer, system_role):
    def _map(example):
        messages = to_chat_messages(example, system_role)
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}
    return dataset.map(_map, remove_columns=dataset.column_names)

## 4. Penghitungan Token

In [9]:
# Hitung distribusi panjang token (min/mean/p50/p95/p99/max) dari dataset yang sudah diformat
def token_length_stats(formatted_dataset, tokenizer):
    lengths = []
    for row in formatted_dataset:
        ids = tokenizer(row["text"], add_special_tokens=False)["input_ids"]
        lengths.append(len(ids))
    lengths = np.array(lengths)
    return {
        "min": int(lengths.min()),
        "mean": float(lengths.mean()),
        "p50": int(np.percentile(lengths, 50)),
        "p95": int(np.percentile(lengths, 95)),
        "p99": int(np.percentile(lengths, 99)),
        "max": int(lengths.max()),
    }
 
 
# Bulatkan max_seq_length ke kelipatan 2^n terdekat di atas nilai token maksimum
def recommend_max_seq_length(stats):
    candidates = [512, 768, 1024, 1536, 2048, 3072, 4096, 6144, 8192]
    target = stats["max"]
    for c in candidates:
        if c >= target:
            return c
    return candidates[-1]

In [10]:
# Tentukan pola delimiter instruction_part/response_part sesuai arsitektur (untuk train_on_responses_only)
def get_response_delimiter(system_role):
    if system_role:
        return "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return "<start_of_turn>user\n", "<start_of_turn>model\n"
 
 
# Cek apakah pola delimiter benar-benar muncul di sample teks hasil format
def validate_mask_pattern(formatted_dataset, instruction_part, response_part, n_samples=5):
    results = []
    for i in range(min(n_samples, len(formatted_dataset))):
        text = formatted_dataset[i]["text"]
        results.append({
            "index": i,
            "instruction_found": instruction_part in text,
            "response_found": response_part in text,
        })
    return results

### VRAM DRY-RUN (subprocess terisolasi per kandidat batch size)

In [11]:
DRY_RUN_SCRIPT = """
import sys, json, torch
from unsloth import FastLanguageModel
 
model_name, max_seq_length, lora_r, batch_size, hf_token = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]), sys.argv[5]
 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True, token=hf_token,
)
model = FastLanguageModel.get_peft_model(
    model, r=lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_r * 2, lora_dropout=0.1, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
text_batch = ["dummy input " * (max_seq_length // 3)] * batch_size
dummy_input = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=max_seq_length, padding=True).to(model.device)
torch.cuda.reset_peak_memory_stats()
autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
with torch.autocast(device_type="cuda", dtype=autocast_dtype):
    out = model(**dummy_input, labels=dummy_input["input_ids"])
    out.loss.backward()
peak_gb = torch.cuda.max_memory_reserved() / 1e9
print(json.dumps({"peak_gb": peak_gb}))
"""

In [12]:
# Jalankan satu percobaan VRAM dry-run di subprocess terpisah, kembalikan peak_gb atau None jika gagal
def estimate_vram_dry_run_subprocess(model_name, max_seq_length, lora_r, batch_size, timeout=600):
    import subprocess, tempfile, os
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(DRY_RUN_SCRIPT)
        script_path = f.name
    try:
        result = subprocess.run(
            ["python", script_path, model_name, str(max_seq_length), str(lora_r), str(batch_size), HF_TOKEN],
            capture_output=True, text=True, timeout=timeout,
        )
        if result.returncode != 0:
            return None
        last_line = result.stdout.strip().split("\n")[-1]
        return json.loads(last_line)["peak_gb"]
    finally:
        os.remove(script_path)

In [13]:
# Coba beberapa kandidat batch size, tiap percobaan diisolasi subprocess sendiri, hentikan saat gagal
def scan_batch_size(model_name, max_seq_length, lora_r, batch_size_candidates):
    vram_results = {}
    for bs in batch_size_candidates:
        peak_gb = estimate_vram_dry_run_subprocess(model_name, max_seq_length, lora_r, bs)
        vram_results[bs] = peak_gb
        if peak_gb is None:
            break
    return vram_results

## 5. Main program

In [14]:
# Jalankan seluruh tahap NB08 untuk satu model: format, statistik token, validasi mask, scan VRAM
def process_model(key, cfg, train_ds, val_ds, batch_size_candidates):
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_model"], token=HF_TOKEN)
 
    formatted_train = format_dataset(train_ds, tokenizer, cfg["system_role"])
    formatted_val = format_dataset(val_ds, tokenizer, cfg["system_role"])
 
    stats = token_length_stats(formatted_train, tokenizer)
    max_seq_length = recommend_max_seq_length(stats)
 
    instruction_part, response_part = get_response_delimiter(cfg["system_role"])
    mask_check = validate_mask_pattern(formatted_train, instruction_part, response_part)
 
    vram_results = scan_batch_size(cfg["hf_model"], max_seq_length, lora_r=16, batch_size_candidates=batch_size_candidates)
 
    formatted_train.save_to_disk(f"{OUTPUT_DIR}/{key}_formatted_train")
    formatted_val.save_to_disk(f"{OUTPUT_DIR}/{key}_formatted_val")
    formatted_train.push_to_hub(HF_REPO, config_name=f"{key}_formatted_train", token=HF_TOKEN)
    formatted_val.push_to_hub(HF_REPO, config_name=f"{key}_formatted_val", token=HF_TOKEN)
 
    return {
        "token_stats": stats,
        "max_seq_length": max_seq_length,
        "instruction_part": instruction_part,
        "response_part": response_part,
        "mask_validation": mask_check,
        "vram_estimate_gb": vram_results,
    }
 
 
# Orkestrasi penuh: load dataset sekali, loop tiap model kandidat, kumpulkan semua hasil
def run_pipeline(batch_size_candidates=(1, 2, 4, 8)):
    train_ds = load_split("train")
    val_ds = load_split("validation")
 
    all_results = {}
    for key, cfg in MODEL_CONFIGS.items():
        all_results[key] = process_model(key, cfg, train_ds, val_ds, batch_size_candidates)
    return all_results

In [15]:
# Jalankan seluruh tahap NB08 untuk satu model: format, statistik token, validasi mask, scan VRAM
def process_model(key, cfg, train_ds, val_ds, batch_size_candidates):
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_model"], token=HF_TOKEN)
 
    formatted_train = format_dataset(train_ds, tokenizer, cfg["system_role"])
    formatted_val = format_dataset(val_ds, tokenizer, cfg["system_role"])
 
    stats = token_length_stats(formatted_train, tokenizer)
    max_seq_length = recommend_max_seq_length(stats)
 
    instruction_part, response_part = get_response_delimiter(cfg["system_role"])
    mask_check = validate_mask_pattern(formatted_train, instruction_part, response_part)
 
    vram_results = scan_batch_size(cfg["hf_model"], max_seq_length, lora_r=16, batch_size_candidates=batch_size_candidates)
 
    formatted_train.save_to_disk(f"{OUTPUT_DIR}/{key}_formatted_train")
    formatted_val.save_to_disk(f"{OUTPUT_DIR}/{key}_formatted_val")
    formatted_train.push_to_hub(HF_REPO, config_name=f"{key}_formatted_train", token=HF_TOKEN)
    formatted_val.push_to_hub(HF_REPO, config_name=f"{key}_formatted_val", token=HF_TOKEN)
 
    return {
        "token_stats": stats,
        "max_seq_length": max_seq_length,
        "instruction_part": instruction_part,
        "response_part": response_part,
        "mask_validation": mask_check,
        "vram_estimate_gb": vram_results,
    }
 
 
# Orkestrasi penuh: load dataset sekali, loop tiap model kandidat, kumpulkan semua hasil
def run_pipeline(batch_size_candidates=(1, 2, 4, 8)):
    all_results = {}
    for key, cfg in MODEL_CONFIGS.items():
        all_results[key] = process_model(key, cfg, train_ds, val_ds, batch_size_candidates)
    return all_results

### Simpan dan Upload config

In [16]:
# Simpan hasil akhir (max_seq_length, delimiter, estimasi VRAM) ke file JSON lokal
def save_training_config(results):
    config = {}
    for key, res in results.items():
        config[key] = {
            "max_seq_length": res["max_seq_length"],
            "instruction_part": res["instruction_part"],
            "response_part": res["response_part"],
            "vram_estimate_gb": res["vram_estimate_gb"],
        }
    output_path = f"{OUTPUT_DIR}/training_config.json"
    with open(output_path, "w") as f:
        json.dump(config, f, indent=2)
    return output_path
 
 
# Upload training_config.json ke HF Hub agar bisa diakses dari RunPod nanti
def push_config_to_hub(config_path):
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=config_path,
        path_in_repo="training_config.json",
        repo_id=HF_REPO,
        repo_type="dataset",
    )
 

In [17]:
# ==== EKSEKUSI ====
results = run_pipeline()
config_path = save_training_config(results)
push_config_to_hub(config_path)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


## 6. Ringkasan

In [18]:
import pandas as pd
 
summary_rows = []
for key, res in results.items():
    valid_batches = [bs for bs, gb in res["vram_estimate_gb"].items() if gb is not None]
    summary_rows.append({
        "model": key,
        "token_min": res["token_stats"]["min"],
        "token_mean": round(res["token_stats"]["mean"], 1),
        "token_p95": res["token_stats"]["p95"],
        "token_p99": res["token_stats"]["p99"],
        "token_max": res["token_stats"]["max"],
        "max_seq_length": res["max_seq_length"],
        "max_batch_size_ok": max(valid_batches) if valid_batches else None,
        "mask_valid": all(m["instruction_found"] and m["response_found"] for m in res["mask_validation"]),
    })

In [19]:
summary_df = pd.DataFrame(summary_rows)
summary_df

,model,token_min,token_mean,token_p95,token_p99,token_max,max_seq_length,max_batch_size_ok,mask_valid
0,llama3.2,152,377.9,648,862,1104,1536,2,True
1,gemma2,98,266.6,475,645,915,1024,4,True
